In [ ]:
from moabb.datasets import PhysionetMI, Weibo2014
from moabb.paradigms import MotorImagery
from moabb.evaluations import WithinSessionEvaluation
from moabb.datasets.utils import find_intersecting_channels

from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

from mne.decoding import CSP
from brainbot_dataset import get_brainbot_dataset
from datasets16 import PhysionetMI16, Weibo2014_16
from custom_models.cnn import CNN

import moabb
import mne

moabb.set_log_level('ERROR')
mne.set_log_level('ERROR')

SUBJECTS = 1
MAX_TRIALS = 1

brainbot_dataset = get_brainbot_dataset()
brainbot_dataset.n_sessions = min(MAX_TRIALS, brainbot_dataset.n_sessions)
brainbot_dataset.subject_list = brainbot_dataset.subject_list[:SUBJECTS]
physionet_dataset = PhysionetMI()
physionet_dataset.subject_list = physionet_dataset.subject_list[:SUBJECTS]
physionet16_dataset = PhysionetMI16()
physionet16_dataset.subject_list = physionet16_dataset.subject_list[:SUBJECTS]
weibo2014_dataset = Weibo2014()
weibo2014_dataset.subject_list = weibo2014_dataset.subject_list[:SUBJECTS]
weibo2014_16_dataset = Weibo2014_16()
weibo2014_16_dataset.subject_list = weibo2014_16_dataset.subject_list[:SUBJECTS]
assert len(weibo2014_16_dataset.subject_list) == SUBJECTS
assert len(weibo2014_dataset.subject_list) == SUBJECTS
assert len(physionet16_dataset.subject_list) == SUBJECTS
assert len(physionet_dataset.subject_list) == SUBJECTS
assert len(brainbot_dataset.subject_list) == min(len(brainbot_dataset.subject_list), SUBJECTS)
assert brainbot_dataset.n_sessions == MAX_TRIALS


datasets = [brainbot_dataset, physionet16_dataset, physionet_dataset, weibo2014_dataset, weibo2014_16_dataset]
dataset_results = {}
dataset_events = ["left_hand", "right_hand", "feet", "hands", "rest"]
sampling = 160 # based on Physionet sampling rate 

electrodes, datasets = find_intersecting_channels(datasets)
print("Datasets used:", [type(d).__name__ for d in datasets])
print("Used electrodes:", electrodes)

paradigm = MotorImagery(n_classes=len(dataset_events), events=dataset_events, resample=sampling)

pipelines = {}

# Base classifiers and preprocessing
svm = OneVsRestClassifier(SVC(kernel='rbf', probability=True))
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

pipelines['CSP + SVM'] = make_pipeline(csp, svm)
pipelines['CSP + LDA'] = make_pipeline(CSP(n_components=8), LinearDiscriminantAnalysis())

# TGSP (Riemannian) pipeline
pipelines['TGSP + SVM'] = make_pipeline(Covariances("oas"), TangentSpace(metric="riemann"), SVC(kernel="linear", probability=True))

# # Custom CNN pipeline // tensorflow models does not work well with moabb n_jobs>1 - crashes observed due to overallocating memory
# pipelines['CNN'] = CNN(sfreq=sampling)

evaluation = WithinSessionEvaluation(paradigm=paradigm, datasets=datasets, overwrite=True, n_jobs=-1)
evaluation.process(pipelines)
results = evaluation.get_results()

Searching dataset: BrainBotDataset
Searching dataset: PhysionetMI16
Searching dataset: PhysionetMI
Searching dataset: Weibo2014
Searching dataset: Weibo2014_16
Datasets used: ['BrainBotDataset', 'PhysionetMI16', 'PhysionetMI', 'Weibo2014', 'Weibo2014_16']
Used electrodes: ['CPz', 'FCz', 'CP4', 'FC3', 'Pz', 'CP1', 'C3', 'FC2', 'C4', 'Cz', 'FC4', 'CP3', 'C2', 'CP2', 'FC1', 'C1']


BrainBot-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 3.6 (2.2e-16 eps * 16 dim * 1e+15  max singular value)
    Estimated rank (data): 16
    data: rank 16 computed from 16 data channels with 0 projectors
    Using tolerance 3.6 (2.2e-16 eps * 16 dim * 1e+15  max singular value)
    Using tolerance 3.6 (2.2e-16 eps * 16 dim * 1e+15  max singular value)
Reducing data rank from 16 -> 16
Estimating class=0 covariance using EMPIRICAL
    Estimated rank (data): 16
Done.
    data: rank 16 computed from 16 data channels with 0 projectors
    Estimated rank (data): 16
    data: rank 16 computed from 16 data channels with 0 projectors
    Using tolerance 3.5 (2.2e-16 eps * 16 dim * 1e+15  max singular value)
    Estimated rank (data): 16
    data: rank 16 computed from 16 data chann

PhysionetMotorImagery16-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 61 (2.2e-16 eps * 16 dim * 1.7e+16  max singular value)
    Estimated rank (data): 16
    data: rank 16 computed from 16 data channels with 0 projectors
Reducing data rank from 16 -> 16
Estimating class=0 covariance using EMPIRICAL
Done.
    Using tolerance 61 (2.2e-16 eps * 16 dim * 1.7e+16  max singular value)
    Using tolerance 61 (2.2e-16 eps * 16 dim * 1.7e+16  max singular value)
Estimating class=1 covariance using EMPIRICAL
    Estimated rank (data): 16
Done.
    data: rank 16 computed from 16 data channels with 0 projectors
    Using tolerance 62 (2.2e-16 eps * 16 dim * 1.7e+16  max singular value)
    Using tolerance 62 (2.2e-16 eps * 16 dim * 1.7e+16  max singular value)
    Estimated rank (data): 16
    data: 

...

Weibo2014-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.9e+16  max singular value)
    Using tolerance 3.9e+02 (2.2e-16 eps * 60 dim * 2.9e+16  max singular value)
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.8e+16  max singular value)
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.8e+16  max singular value)
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.9e+16  max singular value)
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 compute

Weibo2014_16-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.8e+16  max singular value)
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.9e+16  max singular value)
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.9e+16  max singular value)
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.9e+16  max singular value)
    Using tolerance 3.8e+02 (2.2e-16 eps * 60 dim * 2.8e+16  max singular value)
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 compute

Weibo2014_16-WithinSession: 100%|██████████| 1/1 [00:11<00:00, 11.18s/it]


In [20]:
print("Results Summary:")
summary = results.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count'])
summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
print(summary.to_string())
print("=" * 50)

print("\nDetailed Results by Subject and Dataset:")
detailed = results.pivot_table(
    index=['dataset', 'subject', 'session'], 
    columns='pipeline', 
    values='score'
)
print(detailed.round(3).to_string())
print("=" * 50)

Results Summary:
                                     mean  std  count
pipeline   dataset                                   
CSP + LDA  BrainBot                 0.395  NaN      1
           PhysionetMotorImagery    0.597  NaN      1
           PhysionetMotorImagery16  0.598  NaN      1
           Weibo2014                0.472  NaN      1
           Weibo2014_16             0.500  NaN      1
CSP + SVM  BrainBot                 0.467  NaN      1
           PhysionetMotorImagery    0.522  NaN      1
           PhysionetMotorImagery16  0.545  NaN      1
           Weibo2014                0.475  NaN      1
           Weibo2014_16             0.462  NaN      1
TGSP + SVM BrainBot                 0.467  NaN      1
           PhysionetMotorImagery    0.672  NaN      1
           PhysionetMotorImagery16  0.598  NaN      1
           Weibo2014                0.695  NaN      1
           Weibo2014_16             0.692  NaN      1

Detailed Results by Subject and Dataset:
pipeline               

### Test all builtin pipelines

In [ ]:
from moabb import benchmark
import os

moabb_pipelines_path = os.path.join(os.path.dirname(os.path.dirname(moabb.__file__)), "pipelines")
print(moabb_pipelines_path)

results = benchmark(
    pipelines=moabb_pipelines_path,
    evaluations=["WithinSession"],
    paradigms=["MotorImagery"],
    include_datasets=datasets,
    results="./results/",
    overwrite=True,
    plot=True,
    output="./benchmark/",
    n_jobs=-1,
)

In [ ]:
from moabb.analysis.results import Results
from moabb.evaluations import WithinSessionEvaluation
from moabb.paradigms import MotorImagery

# Load existing results  
results = Results(  
    evaluation_class=WithinSessionEvaluation,  
    paradigm_class=MotorImagery,  
    hdf5_path="./results"
)

# Convert to DataFrame
df = results.to_dataframe()  


print("Results Summary:")
summary = df.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count']).reset_index()
summary['pipeline_max_mean'] = summary.groupby('pipeline')['mean'].transform('max')
summary = (summary.sort_values(['pipeline_max_mean', 'mean'], ascending=[False, False])
           .drop(columns='pipeline_max_mean')
           .set_index(['pipeline', 'dataset']))
summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
print(summary.to_string())
print("=" * 50)

print("\nDetailed Results by Subject and Dataset:")
detailed = df.pivot_table(
    index=['dataset', 'subject', 'session'], 
    columns='pipeline', 
    values='score'
)
print(detailed.round(3).to_string())
print("=" * 50)